## Import packages

In [1]:
import os
import sys
import json
import argparse
import numpy as np
import math
from einops import rearrange
import time
import random
import string
import h5py
from tqdm import tqdm
import webdataset as wds
import gc

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

## Configuration

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
data_type = torch.float16 # change depending on your mixed_precision
num_devices = torch.cuda.device_count()
batch_size = 32
num_epochs = 32

print(f"device={device}, data_type={data_type}, num_devices={num_devices}, batch_size={batch_size}, num_epochs={num_epochs}\n")

data_path = "/home/brain-image-filtering/nsd"
subj = 1
subj_list = [subj]
num_sessions = 10
num_test = 2770
num_voxels_list = []

num_samples_per_epoch = (750 * num_sessions) // num_devices
num_iterations_per_epoch = num_samples_per_epoch // (batch_size * len(subj_list))

def my_split_by_node(urls): 
    return urls

print(f"data_path={data_path}, subj={subj}, subj_list={subj_list}, \n\
num_sessions={num_sessions}, num_test={num_test}, num_voxels_list={num_voxels_list}, \n\
num_iterations_per_epoch={num_iterations_per_epoch}")

device=cuda, data_type=torch.float16, num_devices=1, batch_size=32, num_epochs=32

data_path=/home/brain-image-filtering/nsd, subj=1, subj_list=[1], 
num_sessions=10, num_test=2770, num_voxels_list=[], 
num_iterations_per_epoch=234


## Creating wds dataloader

In [3]:
train_data = {}
train_dl = {}
num_voxels = {}
voxels = {}

for s in subj_list:

    train_url = f"{data_path}/wds/subj0{s}/train/" + "{0.." + f"{num_sessions-1}" + "}.tar"
    
    train_data[f'subj0{s}'] = wds.WebDataset(train_url,resampled=True,nodesplitter=my_split_by_node)\
                        .shuffle(750, initial=1500, rng=random.Random(42))\
                        .decode("torch")\
                        .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                        .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])

    train_dl[f'subj0{s}'] = torch.utils.data.DataLoader(train_data[f'subj0{s}'], batch_size=batch_size, shuffle=False, drop_last=True, pin_memory=True)

    f = h5py.File(f'{data_path}/betas_all_subj0{s}_fp32_renorm.hdf5', 'r')
    betas = f['betas'][:]
    betas = torch.Tensor(betas).to("cpu").to(data_type)
    num_voxels_list.append(betas[0].shape[-1])
    num_voxels[f'subj0{s}'] = betas[0].shape[-1]
    voxels[f'subj0{s}'] = betas

    print(f"num_voxels for subj0{s}: {num_voxels[f'subj0{s}']}\n")

print("Loaded all subj train dls and betas!\n")

test_url = f'{data_path}/wds/subj01/test/0.tar'

test_data = wds.WebDataset(test_url,resampled=False,nodesplitter=my_split_by_node)\
                    .shuffle(750, initial=1500, rng=random.Random(42))\
                    .decode("torch")\
                    .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                    .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])

test_dl = {}
test_dl['subj01'] = torch.utils.data.DataLoader(test_data, batch_size=4)

print(f"Loaded test dl for subj{subj}!\n")

num_voxels for subj01: 15724

Loaded all subj train dls and betas!

Loaded test dl for subj1!



In [4]:
f = h5py.File(f'{data_path}/coco_images_224_float16.hdf5', 'r')
# images = f['images'][:] # if you go OOM you can remove the [:] so it isnt preloaded to cpu! (will require a few edits elsewhere though)
# images = torch.Tensor(images).to(device).to(data_type)
images = f['images']
images.shape

(73000, 3, 224, 224)

## Load models

### CLIP image embeddings model

In [5]:
import clip
from PIL import Image

# clip_embedder, preprocess = clip.load("ViT-B/32", device=device)
# input_dim = 512
clip_embedder, preprocess = clip.load("ViT-L/14", device=device)
input_dim = 768

### ImageToBrain

In [6]:
# Input: Clip latents of image
# Output: Brain representation

class ImageToBrain(torch.nn.Module):

    def __init__(self, input_sizes, out_features):
        super(ImageToBrain, self).__init__()
        self.out_features = out_features
        self.linears = torch.nn.Linear(input_sizes, out_features)
    def forward(self, x):
        out = self.linears(x)
        return out

In [7]:
output_dim = num_voxels[f'subj0{subj}']  # Number of voxels as target output
model = ImageToBrain(input_dim, output_dim).to(device)

## Main

### Preprocess

In [8]:
train_dls = [train_dl[f'subj0{s}'] for s in subj_list]
test_dls = [test_dl[f'subj0{s}'] for s in subj_list]

In [9]:
def pad_batch(batch, target_size, padding_value=0):
    # Calculate how much padding is needed
    padding_needed = target_size - batch.size(0)
    if padding_needed > 0:
        # Create a padding tensor of the same dimension except for the batch size dimension
        padding_tensor = torch.zeros((padding_needed,) + batch.shape[1:], dtype=batch.dtype, device=batch.device)
        # Append the padding tensor to the original batch
        batch = torch.cat([batch, padding_tensor], dim=0)
    return batch

In [10]:
image_transforms = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])

def preprocess(dls):
    voxel_iters = {} # empty dict because diff subjects have differing # of voxels
    image_iters = torch.zeros(num_iterations_per_epoch, batch_size*len(subj_list), 3, 224, 224).float()
    
    for s, dl in enumerate(dls):
        with torch.cuda.amp.autocast(dtype=data_type):
            iter = -1
            for behav0, past_behav0, future_behav0, old_behav0 in dl:
                # Load images to cpu from hdf5 (requires sorted indexing)
                image_idx = behav0[:,0,0].cpu().long().numpy()
                image0, image_sorted_idx = np.unique(image_idx, return_index=True)
                if len(image0) != len(image_idx):  # hdf5 cant handle duplicate indexing
                    continue
                iter += 1

                # Ensure indices are sorted
                image_sorted_idx = np.sort(image_sorted_idx)

                image0 = images[image_sorted_idx]
                image0 = torch.tensor(image0, dtype=torch.float16, device=device)  # Convert to tensor
                # Apply transformations on the fetched images
                image0 = image_transforms(image0)  # Apply resizing and normalization
                image0 = pad_batch(image0, 32)                
                image_iters[iter, s*batch_size:s*batch_size+batch_size] = image0
                
                # Similar process for voxel indices
                voxel_idx = behav0[:,0,5].cpu().long().numpy()
                voxel_sorted_idx = voxel_idx[image_sorted_idx]  # Apply the same sorted indices
                voxel_sorted_idx = np.sort(voxel_sorted_idx)  # Ensure voxel indices are sorted
                voxel0 = voxels[f'subj0{subj_list[s]}'][voxel_sorted_idx]
                voxel0 = torch.Tensor(voxel0).unsqueeze(1)
                voxel0 = pad_batch(voxel0, 32)

                voxel_iters[f"subj0{subj_list[s]}_iter{iter}"] = voxel0

                if iter >= num_iterations_per_epoch-1:
                    print(f"iter is greater than or equal to num_iterations_per_epoch-1")
                    break
    return voxel_iters, image_iters

train_voxel_iters, train_image_iters = preprocess(train_dls)
print(f"stored train_voxel_iters and train_image_iters")
test_voxel_iters, test_image_iters = preprocess(test_dls)
print(f"stored test_voxel_iters and test_image_iters")

/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


iter is greater than or equal to num_iterations_per_epoch-1
stored train_voxel_iters and train_image_iters
iter is greater than or equal to num_iterations_per_epoch-1
stored test_voxel_iters and test_image_iters


### Correlation
[Refer mindeye2 code](https://github.com/MedARC-AI/MindEyeV2/blob/1bd6f6c9d26c316a02823c73083e91da52cd05b0/src/utils.py#L64)

In [11]:
def batchwise_pearson_correlation(Z, B):
    # Calculate means
    Z_mean = torch.mean(Z, dim=1, keepdim=True)
    B_mean = torch.mean(B, dim=1, keepdim=True)

    # Subtract means
    Z_centered = Z - Z_mean
    B_centered = B - B_mean

    # Calculate Pearson correlation coefficient
    numerator = Z_centered @ B_centered.T
    Z_centered_norm = torch.linalg.norm(Z_centered, dim=1, keepdim=True)
    B_centered_norm = torch.linalg.norm(B_centered, dim=1, keepdim=True)
    denominator = Z_centered_norm @ B_centered_norm.T

    pearson_correlation = (numerator / denominator)
    return pearson_correlation

### Train

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()  # Mean Squared Error Loss

test_image, test_voxel = None, None
train_loss, batch_pearson, average_test_loss = None, None, None

for epoch in range(num_epochs):
    model.train() # train
    for train_i in range(num_iterations_per_epoch):
        with torch.cuda.amp.autocast(dtype=data_type):
            optimizer.zero_grad()
            train_loss = 0.

            voxel_list = [train_voxel_iters[f"subj0{s}_iter{train_i}"].detach().to(device) for s in subj_list]
            voxel_batch = torch.stack(voxel_list)
            voxel_batch = voxel_batch.view(-1, voxel_batch.shape[-1])

            image = train_image_iters[train_i].detach()
            image = image.to(device)
            clip_latent = clip_embedder.encode_image(image)
            
            voxel_ridge = model(clip_latent)
            
            train_loss = criterion(voxel_ridge, voxel_batch)            
            pearson_corr = batchwise_pearson_correlation(voxel_ridge, voxel_batch)
            batch_pearson = pearson_corr.diagonal().mean().item()
            
            train_loss.backward()
            optimizer.step()
            # print(f"voxel_batch.shape: {voxel_batch.shape}, voxel_ridge.shape: {voxel_ridge.shape}, image.shape: {image.shape}")

    print({"train loss": train_loss.item(), "batch pearson correlation": batch_pearson})

    model.eval() # eval
    test_loss = 0.0
    num_batches = 0  # Track number of batches for averaging loss
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=data_type):  # No gradients needed
    # Assuming 'test_voxel_iters' and 'test_image_iters' are prepared similarly to train variants
        for test_i in range(num_iterations_per_epoch):
            voxel_list = [test_voxel_iters[f"subj0{s}_iter{test_i}"].detach().to(device) for s in subj_list]
            voxel_batch = torch.stack(voxel_list)
            voxel_batch = voxel_batch.view(-1, voxel_batch.shape[-1])

            image = test_image_iters[test_i].detach()
            image = image.to(device)
            clip_latent = clip_embedder.encode_image(image)

            voxel_ridge = model(clip_latent)

            loss = criterion(voxel_ridge, voxel_batch)
            test_loss += loss.item()

            # print(f"Test loss: {loss.item()}")
            # print(f"voxel_batch.shape: {voxel_batch.shape}, voxel_ridge.shape: {voxel_ridge.shape}, image.shape: {image.shape}")

            num_batches += 1
        average_test_loss = test_loss / num_batches

    print({"test loss": loss.item(), "average test loss": average_test_loss})

{'train loss': 1.0002787113189697, 'batch pearson correlation': 0.131591796875}
{'test loss': 0.21153073012828827, 'average test loss': 0.22235980820961487}
{'train loss': 0.9922566413879395, 'batch pearson correlation': 0.1534423828125}
{'test loss': 0.20836089551448822, 'average test loss': 0.2195401462352174}
{'train loss': 0.9904078245162964, 'batch pearson correlation': 0.1588134765625}
{'test loss': 0.20766562223434448, 'average test loss': 0.21891429988492248}
{'train loss': 0.9896955490112305, 'batch pearson correlation': 0.160888671875}
{'test loss': 0.20744125545024872, 'average test loss': 0.21870952889195874}
{'train loss': 0.9893479943275452, 'batch pearson correlation': 0.1619873046875}
{'test loss': 0.20735007524490356, 'average test loss': 0.21862716094041482}
{'train loss': 0.9891567230224609, 'batch pearson correlation': 0.1624755859375}
{'test loss': 0.2073088437318802, 'average test loss': 0.21859115470423657}
{'train loss': 0.9890424609184265, 'batch pearson correl